# Create Ludo Game Dataset

## Overview
This notebook generates a synthetic Ludo game dataset for training a machine learning model to predict winning combinations.

## What Was Done
- Simulated **1,000+ Ludo games** for 4 players: **Red, Green, Yellow, and Blue**
- Each player controls **4 tokens**, following standard Ludo rules:
  - All tokens begin in the home yard (**position 0**); a dice roll of **6** is required to move a token onto the board (**position 1**)
  - Active tokens advance by the dice value; overshooting position **56** causes a **bounce-back**
  - Landing on an opponent's active token **captures** it and sends it back to the home yard
  - Rolling a **6** or making a **capture** grants the player an extra turn
  - A player **wins** when all 4 of their tokens have reached **position 57** (finished)
- Simulation continues until at least **10,000 rows** of data are collected
- Each row records one turn for one player with full 4-token state information
- The final dataset is saved as `ludo_dataset.csv` in the `data file/Raw_Data/` directory

## Output
| Column | Description |
|---|---|
| `Game_ID` | Unique identifier for each simulated game |
| `Turn` | Global turn number within the game |
| `Player` | Player colour (Red, Green, Yellow, Blue) |
| `Dice_Roll` | Value rolled on the dice (1–6) |
| `Token_Moved` | Which token (1–4) was moved; 0 if no valid move was available |
| `Position_Before` | Token position before the move (0 = home, 1–56 = board, 57 = finished; NaN if no move) |
| `Position_After` | Token position after the move (same scale; NaN if no move) |
| `Tokens_Home` | Number of this player's tokens still in the home yard |
| `Tokens_Active` | Number of this player's tokens currently on the board |
| `Tokens_Finished` | Number of this player's tokens that have completed the circuit |
| `Captured_Opponent` | 1 if the move captured an opponent token, 0 otherwise |
| `Is_Winner` | 1 if this player won the game, 0 otherwise |


In [ ]:
# Creating a 4-token-per-player Ludo game dataset for training a ML model
import random
import numpy as np
import pandas as pd
import os

HOME = 0     # Token not yet on the board
FINISH = 57  # Token has completed the circuit (landed exactly on position 56)

def simulate_ludo_4token(min_rows=10000):
    all_records = []
    players = ['Red', 'Green', 'Yellow', 'Blue']
    game_id = 0

    while len(all_records) < min_rows:
        # Each player starts with all 4 tokens at home (position 0)
        tokens = {player: [HOME, HOME, HOME, HOME] for player in players}
        turn_number = 0
        winner = None

        while winner is None:
            for player in players:
                extra_turn = True
                while extra_turn:
                    extra_turn = False
                    dice_roll = random.randint(1, 6)
                    turn_number += 1
                    player_tokens = tokens[player]

                    # Tokens that can legally move this roll
                    can_move = [
                        idx for idx, pos in enumerate(player_tokens)
                        if pos != FINISH and (pos != HOME or dice_roll == 6)
                    ]

                    tokens_home    = player_tokens.count(HOME)
                    tokens_finished = player_tokens.count(FINISH)
                    tokens_active  = 4 - tokens_home - tokens_finished

                    if not can_move:
                        all_records.append({
                            'Game_ID': game_id,
                            'Turn': turn_number,
                            'Player': player,
                            'Dice_Roll': dice_roll,
                            'Token_Moved': 0,
                            'Position_Before': np.nan,
                            'Position_After': np.nan,
                            'Tokens_Home': tokens_home,
                            'Tokens_Active': tokens_active,
                            'Tokens_Finished': tokens_finished,
                            'Captured_Opponent': 0,
                            'Is_Winner': 0
                        })
                        # Rolling a 6 with no valid move still earns an extra turn
                        if dice_roll == 6:
                            extra_turn = True
                        continue

                    # Strategy: prefer tokens already on the board; fall back to home tokens
                    active_movable = [idx for idx in can_move if player_tokens[idx] != HOME]
                    chosen_idx = random.choice(active_movable if active_movable else can_move)
                    pos_before = player_tokens[chosen_idx]

                    # Compute new position
                    if pos_before == HOME:
                        new_pos = 1  # Enter the board
                    else:
                        new_pos_raw = pos_before + dice_roll
                        if new_pos_raw > 56:
                            new_pos = 56 - (new_pos_raw - 56)  # Bounce back
                        elif new_pos_raw == 56:
                            new_pos = FINISH                    # Token finishes!
                        else:
                            new_pos = new_pos_raw

                    # Check for captures (only on active board squares 1–55)
                    captured = 0
                    if 1 <= new_pos <= 55:
                        for other_player in players:
                            if other_player == player:
                                continue
                            for oidx, opos in enumerate(tokens[other_player]):
                                if opos == new_pos:
                                    tokens[other_player][oidx] = HOME
                                    captured = 1

                    player_tokens[chosen_idx] = new_pos
                    tokens[player] = player_tokens

                    tokens_home_after     = player_tokens.count(HOME)
                    tokens_finished_after = player_tokens.count(FINISH)
                    tokens_active_after   = 4 - tokens_home_after - tokens_finished_after

                    all_records.append({
                        'Game_ID': game_id,
                        'Turn': turn_number,
                        'Player': player,
                        'Dice_Roll': dice_roll,
                        'Token_Moved': chosen_idx + 1,
                        'Position_Before': pos_before,
                        'Position_After': new_pos,
                        'Tokens_Home': tokens_home_after,
                        'Tokens_Active': tokens_active_after,
                        'Tokens_Finished': tokens_finished_after,
                        'Captured_Opponent': captured,
                        'Is_Winner': 0
                    })

                    # Win condition: all 4 tokens finished
                    if tokens_finished_after == 4:
                        winner = player
                        break

                    # Extra turn for rolling 6 or capturing an opponent
                    if dice_roll == 6 or captured:
                        extra_turn = True

                if winner:
                    break

            if winner:
                break

        # Mark the winner's records for this game
        for record in all_records:
            if record['Game_ID'] == game_id and record['Player'] == winner:
                record['Is_Winner'] = 1

        game_id += 1

    return all_records

# Generate the dataset (stops once at least 10,000 rows are collected)
ludo_records = simulate_ludo_4token(min_rows=10000)

# Convert to DataFrame
df = pd.DataFrame(ludo_records)

# Save to CSV in the Raw_Data folder
save_path = os.path.join('..', 'data file', 'Raw_Data', 'ludo_dataset.csv')
df.to_csv(save_path, index=False)

print(f"Dataset created with {len(df)} rows across {df['Game_ID'].nunique()} games")
print(f"Saved to: {os.path.abspath(save_path)}")
print(df.head(12))
print("\nWinner distribution:")
winners = df[df['Is_Winner'] == 1].groupby('Player').size()
print(winners)
print(f"\nTotal captures: {int(df['Captured_Opponent'].sum())}")
print(f"\nColumns: {list(df.columns)}")

Dataset created with 10021 rows across 173 games, saved as 'ludo_dataset.csv'
   Game_ID  Turn  Player  Dice_Roll  Position  Is_Winner
0        0     1     Red          1         1          0
1        0     2   Green          5         5          0
2        0     3  Yellow          4         4          0
3        0     4    Blue          6         6          1
4        0     5     Red          1         2          0
5        0     6   Green          4         9          0
6        0     7  Yellow          3         7          0
7        0     8    Blue          6        12          1
8        0     9     Red          2         4          0
9        0    10   Green          2        11          0

Winner distribution:
Player
Blue      651
Green     694
Red       773
Yellow    457
dtype: int64
